In [37]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

from catboost import CatBoostRegressor


In [39]:
train_path = "train.csv"
test_path  = "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(train_df.shape, test_df.shape)
train_df.head()


(15000, 10) (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [40]:
TARGET = "Age"
ID_COL = "id"

assert TARGET in train_df.columns, "do not find target in train!"
assert TARGET not in test_df.columns, "do not find target in test!"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].astype(float)

print("Train columns:", list(train_df.columns))
print("Missing values (train):\n", train_df.isna().sum())
print("dtypes:\n", train_df.dtypes)


Train columns: ['id', 'Sex', 'Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight', 'Age']
Missing values (train):
 id                0
Sex               0
Length            0
Diameter          0
Height            0
Weight            0
Shucked Weight    0
Viscera Weight    0
Shell Weight      0
Age               0
dtype: int64
dtypes:
 id                  int64
Sex                object
Length            float64
Diameter          float64
Height            float64
Weight            float64
Shucked Weight    float64
Viscera Weight    float64
Shell Weight      float64
Age               float64
dtype: object


In [41]:
drop_cols = [ID_COL] if ID_COL in X.columns else []
X = X.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical:", cat_cols)
print("Numeric:", num_cols)


Categorical: ['Sex']
Numeric: ['Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight']


In [42]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [43]:
from sklearn.preprocessing import PolynomialFeatures

# Polynomial features for numeric columns
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)


In [44]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_valid.shape)


(12000, 8) (3000, 8)


In [45]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

cat = CatBoostRegressor(
    loss_function="MAE",
    random_seed=42,
    verbose=0
)


In [54]:
rf_model = RandomForestRegressor(
    n_estimators=1500,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    max_samples=0.9,
    random_state=42,
    n_jobs=-1
)

cat_model = CatBoostRegressor(
    loss_function="MAE",
    iterations=5000,
    learning_rate=0.02,
    depth=10,
    l2_leaf_reg=2.0,
    bootstrap_type='Bernoulli',
    subsample=0.85,
    random_seed=42,
    verbose=0
)

rf_pipe  = Pipeline([("preprocess", preprocess), ("model", rf_model)])
cat_pipe = Pipeline([("preprocess", preprocess), ("model", cat_model)])


In [55]:
from sklearn.ensemble import StackingRegressor

from sklearn.linear_model import Ridge



In [56]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=9,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.5,
    reg_lambda=1.0,
    objective="reg:absoluteerror",
    random_state=42,
    n_jobs=-1
)

lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.5,
    reg_lambda=1.0,
    objective="mae",
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

xgb_pipe = Pipeline([("preprocess", preprocess), ("model", xgb_model)])
lgb_pipe = Pipeline([("preprocess", preprocess), ("model", lgb_model)])


In [ ]:
# 4 models diye stacking - RF, CatBoost, XGBoost, LightGBM
from sklearn.linear_model import ElasticNet

stack = StackingRegressor(
    estimators=[
        ("rf", rf_pipe), 
        ("cat", cat_pipe), 
        ("xgb", xgb_pipe),
        ("lgb", lgb_pipe)
    ],
    final_estimator=ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    n_jobs=-1
)

stack.fit(X_train, y_train)


In [36]:
valid_pred = stack.predict(X_valid)
mae = mean_absolute_error(y_valid, valid_pred)
print("Holdout Validation MAE:", mae)


Holdout Validation MAE: 1.3811255281092245


In [ ]:
# Individual model predictions o check koro
rf_pred = rf_pipe.fit(X_train, y_train).predict(X_valid)
cat_pred = cat_pipe.fit(X_train, y_train).predict(X_valid)
xgb_pred = xgb_pipe.fit(X_train, y_train).predict(X_valid)
lgb_pred = lgb_pipe.fit(X_train, y_train).predict(X_valid)

print("RF MAE:", mean_absolute_error(y_valid, rf_pred))
print("CatBoost MAE:", mean_absolute_error(y_valid, cat_pred))
print("XGBoost MAE:", mean_absolute_error(y_valid, xgb_pred))
print("LightGBM MAE:", mean_absolute_error(y_valid, lgb_pred))
